# Fiorell.IA Azure ML Master Release

This notebook submits the Fiorell.IA LoRA training job to Azure ML, registers the adapter output, deploys it to a managed online endpoint, and writes `azure_deploy_summary.json` locally. Keep the summary out of Git because it can contain an endpoint key.


## 1. Install dependencies

The notebook uses Azure ML Python SDK v2 (`azure-ai-ml`) and Python 3.10+ for managed online endpoint work.


In [ ]:
%pip install -q --upgrade pip
%pip install -q azure-ai-ml azure-identity pyyaml


## 2. Load local project config

Run this notebook from the repository root in VS Code. The dataset path is repository-relative; no Colab `/content/...` paths are used.


In [ ]:
import json
import os
from pathlib import Path

import yaml

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "fiorellia").exists():
            return candidate
    raise RuntimeError("Repository root not found. Open the notebook inside regulatory-insight-engine.")

def expand_env(value):
    if isinstance(value, str):
        return os.path.expandvars(value)
    return value

repo_root = find_repo_root(Path.cwd())
config_path = repo_root / "azure_ml_config.yaml"
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
config = {key: expand_env(value) for key, value in config.items()}

dataset_path = repo_root / config["dataset_path"]
training_config_path = repo_root / config["training_config"]
assert dataset_path.exists(), dataset_path
assert training_config_path.exists(), training_config_path

required = ["subscription_id", "resource_group", "workspace_name"]
missing = [key for key in required if not config.get(key) or str(config[key]).startswith("${")]
if missing:
    raise ValueError(f"Set these environment variables before continuing: {missing}")

print(json.dumps({
    "repo_root": str(repo_root),
    "dataset_path": str(dataset_path),
    "training_config_path": str(training_config_path),
    "workspace": config["workspace_name"],
    "compute": config["compute_name"],
    "endpoint": config["endpoint_name"],
}, indent=2))


## 3. Connect to Azure ML workspace

Use `DefaultAzureCredential`; in VS Code this can use Azure CLI login, VS Code login, managed identity, or interactive browser fallback depending on your environment.


In [ ]:
from azure.ai.ml import Input, MLClient, Output, command
from azure.ai.ml.entities import AmlCompute, CodeConfiguration, Environment, ManagedOnlineDeployment, ManagedOnlineEndpoint, Model
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

try:
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id=config["subscription_id"],
    resource_group_name=config["resource_group"],
    workspace_name=config["workspace_name"],
)
workspace = ml_client.workspaces.get(config["workspace_name"])
print(f"connected_workspace={workspace.name}")


## 4. Ensure compute target


In [ ]:
compute_name = config["compute_name"]
try:
    compute = ml_client.compute.get(compute_name)
    print(f"compute_exists={compute.name}")
except Exception:
    compute = AmlCompute(
        name=compute_name,
        size=config["compute_size"],
        min_instances=int(config.get("compute_min_instances", 0)),
        max_instances=int(config.get("compute_max_instances", 1)),
    )
    poller = ml_client.begin_create_or_update(compute)
    compute = poller.result()
    print(f"compute_created={compute.name}")


## 5. Submit LoRA training job

The command writes the adapter into an Azure ML job output folder named `adapter`, so the next cell can register it as a model artifact.


In [ ]:
training_env = Environment(
    name="fiorellia-lora-training-env",
    image="mcr.microsoft.com/azureml/curated/acpt-pytorch-2.2-cuda12.1:latest",
    conda_file=str(repo_root / "fiorellia" / "training" / "azure_endpoint" / "conda.yaml"),
)
ml_client.environments.create_or_update(training_env)

job = command(
    code=str(repo_root),
    command=(
        "python fiorellia/training/train_lora_behavior_v1.py "
        "--config ${{inputs.training_config}} "
        "--dataset-path ${{inputs.dataset}} "
        "--output-dir ${{outputs.adapter}}"
    ),
    inputs={
        "training_config": Input(type="uri_file", path=str(training_config_path)),
        "dataset": Input(type="uri_file", path=str(dataset_path)),
    },
    outputs={"adapter": Output(type="uri_folder")},
    environment=training_env,
    compute=compute_name,
    experiment_name=config["experiment_name"],
    display_name="fiorellia-qwen25-lora-release",
)

created_job = ml_client.jobs.create_or_update(job)
print(f"job_name={created_job.name}")
ml_client.jobs.stream(created_job.name)


## 6. Register adapter model


In [ ]:
adapter_uri = f"azureml://jobs/{created_job.name}/outputs/{config['adapter_output_name']}"
registered_model = ml_client.models.create_or_update(
    Model(
        path=adapter_uri,
        name=config["model_name"],
        type="custom_model",
        description="Fiorell.IA Qwen2.5-3B LoRA adapter release artifact.",
    )
)
print(f"registered_model={registered_model.name}:{registered_model.version}")


## 7. Deploy managed online endpoint

The scoring code lives in `fiorellia/training/azure_endpoint/score.py` and returns answer, confidence score, and abstention status.


In [ ]:
endpoint = ManagedOnlineEndpoint(
    name=config["endpoint_name"],
    auth_mode=config.get("auth_mode", "key"),
)
endpoint = ml_client.begin_create_or_update(endpoint).result()

deployment_env = Environment(
    name="fiorellia-lora-inference-env",
    image="mcr.microsoft.com/azureml/curated/acpt-pytorch-2.2-cuda12.1:latest",
    conda_file=str(repo_root / "fiorellia" / "training" / "azure_endpoint" / "conda.yaml"),
)
ml_client.environments.create_or_update(deployment_env)

deployment = ManagedOnlineDeployment(
    name=config["deployment_name"],
    endpoint_name=config["endpoint_name"],
    model=registered_model,
    environment=deployment_env,
    code_configuration=CodeConfiguration(
        code=str(repo_root / "fiorellia" / "training" / "azure_endpoint"),
        scoring_script="score.py",
    ),
    instance_type=config["instance_type"],
    instance_count=int(config.get("instance_count", 1)),
)
deployment = ml_client.begin_create_or_update(deployment).result()

endpoint.traffic = {config["deployment_name"]: 100}
endpoint = ml_client.begin_create_or_update(endpoint).result()
print(f"endpoint_uri={endpoint.scoring_uri}")


## 8. Write deployment summary

`azure_deploy_summary.json` is ignored by Git. Copy it to the Google Drive project folder only when you are ready to share runtime credentials through the agreed secure channel.


In [ ]:
keys = ml_client.online_endpoints.get_keys(name=config["endpoint_name"])
summary = {
    "endpoint_name": config["endpoint_name"],
    "deployment_name": config["deployment_name"],
    "endpoint_url": endpoint.scoring_uri,
    "api_key": keys.primary_key,
    "model_name": registered_model.name,
    "model_version": registered_model.version,
    "job_name": created_job.name,
    "generated_at": __import__("datetime").datetime.utcnow().isoformat() + "Z",
}
summary_path = repo_root / config.get("deploy_summary_path", "azure_deploy_summary.json")
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps({**summary, "api_key": "***redacted***"}, indent=2))


## 9. Optional smoke invocation


In [ ]:
sample_request = {"query": "Quali disclosure Pillar 3 specifiche pubblica Intesa Sanpaolo nell'ultimo report disponibile?"}
request_path = repo_root / ".artifacts" / "azure_sample_request.json"
request_path.parent.mkdir(exist_ok=True)
request_path.write_text(json.dumps(sample_request, ensure_ascii=False), encoding="utf-8")
response = ml_client.online_endpoints.invoke(
    endpoint_name=config["endpoint_name"],
    deployment_name=config["deployment_name"],
    request_file=str(request_path),
)
print(response)
